# Starting Preprocessing

In [3]:
# 0.0 Imports

import pandas as pd
import numpy as np

## Phase 1: Basic Cleanup

**GOAL:**
- to reduce the shape from (1783, 39) -> (1470, 28)

In [4]:
# 1.1 Import the raw dataset

df = pd.read_csv(r'D:\Hustle\Chennai-PG\Data\raw\chennai_pg_dataset.csv')
df.shape

(1783, 39)

In [5]:
# 1.2 Deduplication

"""
Drop duplicate rows corresponding to ID & OCCUPANCY
"""

df = df.drop_duplicates(subset=['id', 'occupancy'])
print(df.shape)
print(df.duplicated().sum())

(1621, 39)
0


In [6]:
# 1.3 Drop columns

"""
Drop columns that are not useful for modeling.

If columns is not provided, the default set of columns
identified during EDA will be removed.

col = ['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed'] by phase 1 observation
col = ['lunch', 'breakfast', 'dinner'] by phase 3 observation
"""

df = df.drop(columns=['id', 'title', 'address', 'total_bathrooms', 'warden', 'cooking_allowed', 'gate_closing_time', 'guardian_required', 'nonveg_allowed', 'smoking_allowed', 'lunch', 'breakfast', 'dinner'])
df.shape

(1621, 26)

In [7]:
# 1.4 Drop rows

"""
Drop rows that are not useful for modelling.

- ~1% rows with missing values in key columns
- rent with 0 or NaNs
- occupancy is NaN
- Known confirmed correction for THIS dataset
"""

df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])

# Remove invalid/placeholder rents.
# The minimum realistic PG rent in Chennai is well above 1000.
df = df[df['rent'] >= 1000]

# Known confirmed correction for THIS dataset
df = df[~((df['deposit'] == 200000) & (df['rent'] == 25000) & (df['locality'] == 'Vadapalani'))]
df.shape


(1470, 26)

In [8]:
# 1.5 fix datatypes & renaming

# Boolean cleanup
"""
FIll missing boolean amenites with False
then convert the columns to bool type
"""
bool_cols = ['attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
        'refrigerator', 'common_tv', 'room_cleaning','room_ac', 
        'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
        'room_attached_bath',
]
df[bool_cols] = df[bool_cols].fillna(False).astype(bool)
df['parking'] = df['parking'].fillna('No Parking') # in EDA i actually replaced NaN as none, but it make sense to keep No Parking

# Naming consistency
df['available_for'] = df['available_for'].replace('Both', 'Anyone')


In [9]:
# 1.6 fixes for before doing imputations

# this is for fix the left influend skew-ness (should be done before imputation)
# Convert the invalid sentinel value (-10) to NaN
# before creating missing-value indicators and imputing.
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# creating tag for msiing values rows
df['transit_score_missing'] = df['transit_score'].isna().astype(int)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna().astype(int)

In [8]:
df.shape

(1470, 28)

## Phase 1 Observation

- our goal (**to reduce the shape from (1783, 39) -> (1470, 28)**) was satisfied

----

# Phase 2: Train/Validation/Test split

**SPLIT PLAN**:

```md
100%
│
├── 70% TRAIN
│
├── 15% VALIDATION
│
└── 15% TEST
```

| Dataset        | Purpose                                                          |
| -------------- | ---------------------------------------------------------------- |
| **Train**      | Learn model parameters + fit preprocessing                       |
| **Validation** | Make decisions: model, hyperparameters, features, encoding, etc. |
| **Test**       | Final unbiased evaluation                                        |


In [10]:
# 2.1 Target and feature

X = df.drop(columns=['rent'])
y = df['rent']

print(X.shape)
print(y.shape)

(1470, 27)
(1470,)


In [11]:
# 2.2 Train/Validation/Test

from sklearn.model_selection import train_test_split

# First spilt (Train = 70%, temp = 30%) here i use validation so, just used temp and then splt the temp -> val/test = 15% each
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=X['occupancy'] # self note: Bug Fix (refer commit description)
)

# Second Split (temp = 30%, split it inro Validation/Train -> 15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=X_temp['occupancy'] # self note: Bug Fix (refer commit description)
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1029, 27) (1029,)
Validation: (220, 27) (220,)
Test: (221, 27) (221,)


# Phase 2: Results

- Created feature and target variable
- splitted Train/Validation/test as of the plan

---

# Phase 3: Imputations

**GOAL**:

- apply `log1p` for target (rent)
- similarly apply `log1p` transfromation for deposit
- cast bool_cols to int8 (forgoted and added later) skill issue :(
- Impute transit_score and lifestyle_score as per the Phase 2 (EDA) strategy
- `locality`: Smoothed target encoding // **The rule**: always fit target encoding on train only, then apply to val and test.
- `occupancy`: Ordinal encoding
- `gender`, `parking` & `available_for`: One-Hot encoding

In [12]:
# 3.1 Transform rent and deposit
import numpy as np

# For train 
y_train = np.log1p(y_train)
X_train['deposit'] = np.log1p(X_train['deposit'])

# for validation 
y_val = np.log1p(y_val)
X_val['deposit'] = np.log1p(X_val['deposit'])

# for test 
y_test = np.log1p(y_test)
X_test['deposit'] = np.log1p(X_test['deposit'])

In [13]:
# 3.2 Cast bool -> int8

bool_cols = ['attached_bathroom', 'food_included', 'mess', 'wifi', 'laundry', 'power_backup', 'refrigerator', 'common_tv', 'room_cleaning', 'room_ac', 'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding', 'room_attached_bath']

X_train[bool_cols] = X_train[bool_cols].astype('int8')
X_val[bool_cols] = X_val[bool_cols].astype('int8')
X_test[bool_cols] = X_test[bool_cols].astype('int8')

In [14]:
# 3.3 Transit & lifestle score imptation

# Impute with locality meadian for train/val/test\

# transit_score
X_train['transit_score'] = (
    X_train.groupby('locality')['transit_score']
    .transform(lambda x: x.fillna(x.median()))
)
X_val['transit_score'] = (
    X_val.groupby('locality')['transit_score']
    .transform(lambda x: x.fillna(x.median()))
)
X_test['transit_score'] = (
    X_test.groupby('locality')['transit_score']
    .transform(lambda x: x.fillna(x.median()))
)

# lifestyle_score
X_train['lifestyle_score'] = (
    X_train.groupby('locality')['lifestyle_score']
    .transform(lambda x: x.fillna(x.median()))
)
X_val['lifestyle_score'] = (
    X_val.groupby('locality')['lifestyle_score']
    .transform(lambda x: x.fillna(x.median()))
)
X_test['lifestyle_score'] = (
    X_test.groupby('locality')['lifestyle_score']
    .transform(lambda x: x.fillna(x.median()))
)

# impute with global median (if local median is Nan)
X_train['transit_score'] = X_train['transit_score'].fillna(X_train['transit_score'].median())
X_train['lifestyle_score'] = X_train['lifestyle_score'].fillna(X_train['lifestyle_score'].median())

X_val['transit_score'] = X_val['transit_score'].fillna(X_val['transit_score'].median())
X_val['lifestyle_score'] = X_val['lifestyle_score'].fillna(X_val['lifestyle_score'].median())

X_test['transit_score'] = X_test['transit_score'].fillna(X_test['transit_score'].median())
X_test['lifestyle_score'] = X_test['lifestyle_score'].fillna(X_test['lifestyle_score'].median())


In [16]:
print(X_train.isna().sum())
print(X_val.isna().sum())
print(X_test.isna().sum())

# fixed my mistake that i only performed imputation only for train datset , so im just did for val/train also

latitude                   0
longitude                  0
locality                   0
gender                     0
available_for              0
transit_score              0
lifestyle_score            0
occupancy                  0
deposit                    0
attached_bathroom          0
food_included              0
mess                       0
wifi                       0
laundry                    0
power_backup               0
refrigerator               0
common_tv                  0
room_cleaning              0
parking                    0
room_ac                    0
room_cupboard              0
room_tv                    0
room_geyser                0
room_bedding               0
room_attached_bath         0
transit_score_missing      0
lifestyle_score_missing    0
dtype: int64
latitude                   0
longitude                  0
locality                   0
gender                     0
available_for              0
transit_score              0
lifestyle_score            0
o

In [ ]:
# 3.4 Encoding / column transformation

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder

# 3.4.1 fit and transform on train dataset

# Occupancy -> ordinal encoder
ord_enc = OrdinalEncoder(categories=[['SINGLE','DOUBLE', 'THREE', 'FOUR']]) # self note: plain OrdinalEncoder() will assign categories alphabetically, so i need to define the order explicitly

# fit
ord_enc.fit(X_train[['occupancy']]) # DataFrame should be in → 2D, cuz sklearn expexts 2D not series, df['occupancy'] is a series

# transform
occupancy_train = ord_enc.transform(X_train[['occupancy']])
print(f'Ordinal Encoding: \n{ord_enc.categories_}')

# parking, gender & available_for -> onehot encoding
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# fit
ohe.fit(X_train[['gender', 'parking', 'available_for']])

# transform
ohe_train = ohe.transform(X_train[['gender', 'parking', 'available_for']])

ohe.categories_

# locality smoothed target encoding
tar_enc = TargetEncoder(target_type='continuous') # rent is continous so, this rely on target soo

# fit & transform
locality_train = tar_enc.fit_transform(X_train[['locality']], y_train) # x = whcih column to tranform, y = target -> whoch is in y_train soo

Ordinal Encoding: 
[array(['SINGLE', 'DOUBLE', 'THREE', 'FOUR'], dtype=object)]


In [30]:
# 3.4.2 transform on val/test dataset

# Occupancy -> ordinal encoder

occupancy_val = ord_enc.transform(X_val[['occupancy']])
occupancy_test = ord_enc.transform(X_test[['occupancy']])

# parking, gender, available for -> OHE

ohe_val = ohe.transform(X_val[['gender', 'parking', 'available_for']])

ohe_test = ohe.transform(X_test[['gender', 'parking', 'available_for']])

# locality -> target encoded
"""
fit_transform for train to use cross-fitting
transform only for validation/test
"""

locality_val = tar_enc.transform(X_val[['locality']])
locality_test = tar_enc.transform(X_test[['locality']])

In [21]:
X_train.head()

,latitude,longitude,locality,gender,available_for,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,...,room_cleaning,parking,room_ac,room_cupboard,room_tv,room_geyser,room_bedding,room_attached_bath,transit_score_missing,lifestyle_score_missing
1676,13.049858,80.210085,Vadapalani,MALE,Anyone,8.4,8.0,FOUR,5000.0,False,...,True,Bike,False,False,False,False,False,False,0,0
1617,13.023877,80.227445,Saidapet,FEMALE,Anyone,8.5,7.8,FOUR,1000.0,False,...,False,Bike,False,False,False,False,False,False,0,0
1498,12.979389,80.258908,East Coast Road-Thiruvanmiyur,FEMALE,Anyone,7.8,8.4,SINGLE,40000.0,False,...,True,Bike and Car,False,False,False,False,False,False,0,0
638,12.918675,80.233188,Karapakkam,FEMALE,Anyone,2.9,6.7,THREE,3500.0,False,...,False,Bike,True,False,False,False,False,False,0,0
666,12.913037,80.228030,OMR-Karappakam,FEMALE,Anyone,NaN,NaN,FOUR,3000.0,False,...,False,Bike,True,False,False,False,False,False,1,1


# Phase 3: Results:

- **all goals defiend in start of phase 3 were satiesfied**

----
# PHASE 4: final feature assembly

GOAL:

```md
Original numerical columns
        +
Occupancy → ordinal
        +
Gender/Parking/Available_for → OHE
        +
Locality → smoothed target encoding
        ↓
FINAL X_train / X_val / X_test
```

In [27]:
X_train.head()

,latitude,longitude,locality,gender,available_for,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,...,room_cleaning,parking,room_ac,room_cupboard,room_tv,room_geyser,room_bedding,room_attached_bath,transit_score_missing,lifestyle_score_missing
1676,13.049858,80.210085,Vadapalani,MALE,Anyone,8.4,8.0,FOUR,2.253121,False,...,True,Bike,False,False,False,False,False,False,0,0
1617,13.023877,80.227445,Saidapet,FEMALE,Anyone,8.5,7.8,FOUR,2.067970,False,...,False,Bike,False,False,False,False,False,False,0,0
1498,12.979389,80.258908,East Coast Road-Thiruvanmiyur,FEMALE,Anyone,7.8,8.4,SINGLE,2.450717,False,...,True,Bike and Car,False,False,False,False,False,False,0,0
638,12.918675,80.233188,Karapakkam,FEMALE,Anyone,2.9,6.7,THREE,2.214934,False,...,False,Bike,True,False,False,False,False,False,0,0
666,12.913037,80.228030,OMR-Karappakam,FEMALE,Anyone,6.3,6.3,FOUR,2.197969,False,...,False,Bike,True,False,False,False,False,False,1,1


In [46]:
# 4.1 merge transformed (ordinal encoding) occupancy column -> train/test/val splitted datset

X_train['occupancy'] = occupancy_train
X_val['occupancy'] = occupancy_val
X_test['occupancy'] = occupancy_test

# 4.2 merge transformed (OHE) ['gender', 'parking', 'available_for'] column -> train/test/val splitted datset

# get the actual column names OHE created
ohe_cols = ohe.get_feature_names_out(['gender', 'parking', 'available_for'])

# drop the original 3 columns first
X_train = X_train.drop(columns=['gender', 'parking', 'available_for'])
X_val   = X_val.drop(columns=['gender', 'parking', 'available_for'])
X_test  = X_test.drop(columns=['gender', 'parking', 'available_for'])

# add the expanded OHE columns
X_train[ohe_cols] = ohe_train
X_val[ohe_cols]   = ohe_val
X_test[ohe_cols]  = ohe_test

# 4.3 merge transformed (Smoothed target encoding) locality column -> train/test/val splitted datset

X_train['locality'] = locality_train
X_val['locality'] = locality_val
X_test['locality'] = locality_test

In [2]:
X_train.head()

NameError: name 'X_train' is not defined